# TikHarm baseline: video-only, audio-only, simple concatenation

**Rule: baseline before the fancy model.** This notebook trains plain logistic regression on the frozen
embeddings and reports what every later method (late fusion, cross-attention, router) has to beat.

- **Primary set:** the duration-matched subset (balanced, 4 classes, shortcut reduced).
- **Secondary set:** the full clean set (more data, class-weighted).
- **Tuning:** `C` is picked on `val`. `test` is only used for the final numbers.
- **Uncertainty:** every number has a 95% bootstrap interval over the test clips.
- **Reference rows:** metadata-only model (the shortcut floor) and a shuffled-label sanity check.

Nothing here displays clips. It only reads numbers.

In [13]:
import shutil
import os

SRC = "/kaggle/input/datasets/sanawerb/emb-folder/kaggle/working/emb"
EMB = "/kaggle/working/emb"

if os.path.exists(EMB):
    shutil.rmtree(EMB)

shutil.copytree(SRC, EMB)

print("Copied:", EMB)
print(os.listdir(EMB)[:20])

Copied: /kaggle/working/emb
['chunk_024.npz', 'chunk_029.npz', 'chunk_027.npz', 'chunk_008.npz', 'chunk_021.npz', 'chunk_007.npz', 'chunk_028.npz', 'chunk_023.npz', 'chunk_009.npz', 'chunk_020.npz', 'chunk_002.npz', 'chunk_003.npz', 'chunk_006.npz', 'chunk_016.npz', 'chunk_030.npz', 'chunk_014.npz', 'chunk_025.npz', 'chunk_000.npz', 'order.csv', 'chunk_005.npz']


In [14]:
import os, re, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

EMB = os.environ.get("EMB_DIR", "/kaggle/working/emb"); PREP = os.environ.get("PREP_DIR", "/kaggle/input/datasets/sanawerb/preparation")

def creator(path):
    b = os.path.splitext(os.path.basename(path))[0]
    m = re.match(r"^(.*)_(\d{12,20})$", b)
    return (m.group(1) if m else b).lower()

def duration_match(d, bins, seed=0):
    d = d.copy(); d["dbin"] = pd.cut(d.dur, bins, labels=False, include_lowest=True); parts = []
    for (sp, b), g in d.groupby(["split", "dbin"]):
        c = g.label.value_counts()
        if len(c) < 4: continue
        n_ = int(c.min()); parts += [g[g.label == k].sample(n_, random_state=seed) for k in range(4)]
    return pd.concat(parts) if parts else d.iloc[:0]

def regroup(o, m, bins=(0, 10, 20, 40, 90, 1e9), seed=0):
    g = m.path.map(creator).values; y = o.label.values
    rest, test = next(iter(StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(np.zeros(len(o)), y, g)))
    tr_i, va_i = next(iter(StratifiedGroupKFold(8, shuffle=True, random_state=seed + 1).split(np.zeros(len(rest)), y[rest], g[rest])))
    new = np.array(["train"] * len(o), dtype=object); new[rest[va_i]] = "val"; new[test] = "test"
    sets = {s: set(g[new == s]) for s in ("train", "val", "test")}
    assert not (sets["train"] & sets["val"]) and not (sets["train"] & sets["test"]) and not (sets["val"] & sets["test"]), "creator overlap!"
    matched = duration_match(m.assign(uid=o.uid.values, split=new, label=y), list(bins), seed)
    o2 = o.copy(); o2["split"] = new; o2["in_matched"] = o2.uid.isin(matched.uid).astype(int)
    return o2

if not os.path.exists(f"{EMB}/order_official.csv"):
    pd.read_csv(f"{EMB}/order.csv").to_csv(f"{EMB}/order_official.csv", index=False)     # keep the original split
o = pd.read_csv(f"{EMB}/order_official.csv")
m = pd.read_csv(f"{PREP}/manifest_full (1).csv").set_index("uid").loc[o.uid].reset_index()
o2 = regroup(o, m)
o2.to_csv(f"{EMB}/order.csv", index=False)           # same row order as the chunks, new split and matched flags
print(pd.crosstab(o2.cls, o2.split)); print("\nmatched:"); print(pd.crosstab(o2.cls[o2.in_matched == 1], o2.split[o2.in_matched == 1]))

split            test  train  val
cls                              
Adult Content     218    662   68
Harmful Content   182    669  128
Safe              233    684   70
Suicide           238    669   70

matched:
split            test  train  val
cls                              
Adult Content      99    251   26
Harmful Content    99    251   26
Safe               99    251   26
Suicide            99    251   26


In [15]:
# Cell 1: load features and manifests

import os, re, glob, json, time
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    confusion_matrix,
    recall_score
)

# ============================================================
# PATHS
# ============================================================

# IMPORTANT:
# order.csv was rewritten by the creator-grouped split cell
# inside the writable Kaggle working directory.
EMB = os.environ.get(
    "EMB_DIR",
    "/kaggle/working/emb"
)

# Preparation files are attached as a Kaggle input dataset,
# so this location is read-only but fine for reading.
PREP = os.environ.get(
    "PREP_DIR",
    "/kaggle/input/datasets/sanawerb/preparation"
)

WORK = os.environ.get(
    "WORK_DIR",
    "/kaggle/working"
)

RES = f"{WORK}/results"
os.makedirs(RES, exist_ok=True)

SEED, NBOOT = 0, 1000
CLASS_NAMES = ["Safe", "Harmful", "Adult", "Suicide"]


# ============================================================
# CHECK THAT THE EMBEDDING DIRECTORY EXISTS
# ============================================================

assert os.path.exists(EMB), (
    f"Embedding directory not found: {EMB}"
)

assert os.path.exists(f"{EMB}/order.csv"), (
    f"order.csv not found at: {EMB}/order.csv"
)

print("EMB:", EMB)
print("PREP:", PREP)


# ============================================================
# LOAD ORDER
# ============================================================

order = pd.read_csv(
    f"{EMB}/order.csv"
)


# ============================================================
# LOAD EMBEDDING CHUNKS
# ============================================================

files = sorted(
    f
    for f in glob.glob(f"{EMB}/chunk_*.npz")
    if re.search(r"chunk_\d{3}\.npz$", f)
)

assert len(files) > 0, (
    f"No embedding chunks found in {EMB}"
)

parts = [
    np.load(f, allow_pickle=True)
    for f in files
]

F = {
    k: np.concatenate(
        [p[k] for p in parts]
    )
    for k in parts[0].files
}


# ============================================================
# ALIGNMENT CHECK
# ============================================================

assert list(F["uid"]) == list(
    order.uid[:len(F["uid"])]
), "chunks do not match order.csv"

n = len(F["uid"])

order = order.iloc[:n].copy()

print(
    f"{n} clips loaded from {len(files)} chunks "
    f"(full set = {len(pd.read_csv(f'{EMB}/order.csv'))})"
)


# ============================================================
# LOAD FULL MANIFEST
# ============================================================

man = (
    pd.read_csv(
        f"{PREP}/manifest_full (1).csv"
    )
    .set_index("uid")
    .loc[order.uid]
    .reset_index()
)


# ============================================================
# BASIC LABEL / SPLIT ARRAYS
# ============================================================

y = order.label.values

split = order.split.values

in_matched = (
    order.in_matched.values.astype(bool)
)


# ============================================================
# LOAD EMBEDDINGS
# ============================================================

vit = F["vit"].astype(np.float32)

w2v = F["w2v"].astype(np.float32)

ast = F["ast"].astype(np.float32)


# ============================================================
# BUILD FEATURE SETS
# ============================================================

X = {

    "video (ViT mean)":
        vit.mean(1),

    "video (ViT mean+std)":
        np.concatenate(
            [
                vit.mean(1),
                vit.std(1)
            ],
            1
        ),

    "audio (wav2vec2)":
        w2v.mean(1),

    "audio (AST)":
        ast.mean(1),

    "audio (AST probs)":
        F["ast_probs"].astype(np.float32),
}


# ============================================================
# MULTIMODAL FEATURES
# ============================================================

X["video + wav2vec2"] = np.concatenate(
    [
        X["video (ViT mean)"],
        X["audio (wav2vec2)"]
    ],
    1
)

X["video + AST"] = np.concatenate(
    [
        X["video (ViT mean)"],
        X["audio (AST)"]
    ],
    1
)

X["video + AST + wav2vec2"] = np.concatenate(
    [
        X["video (ViT mean)"],
        X["audio (AST)"],
        X["audio (wav2vec2)"]
    ],
    1
)


# ============================================================
# FEATURE DIMENSIONS
# ============================================================

print(
    "Feature dimensions:"
)

print({
    k: v.shape[1]
    for k, v in X.items()
})


# ============================================================
# SPLIT / MATCHED-SET SUMMARY
# ============================================================

print(
    "\nSplit / matched counts:"
)

print(
    pd.crosstab(
        order.cls,
        [
            order.split,
            order.in_matched
        ]
    )
)


# ============================================================
# NEAR-DUPLICATE LEAKAGE CHECK
# ============================================================

# Normalize ViT mean embeddings for cosine similarity.
v = (
    X["video (ViT mean)"]
    /
    (
        np.linalg.norm(
            X["video (ViT mean)"],
            axis=1,
            keepdims=True
        )
        + 1e-8
    )
)

# Test clips vs train/validation clips.
te_i = np.where(
    split == "test"
)[0]

tr_i = np.where(
    split != "test"
)[0]

best = (
    v[te_i]
    @
    v[tr_i].T
).max(1)

leak = te_i[
    best > 0.95
]

split[leak] = "excluded"

print(
    "test clips excluded for near-duplicate leakage:",
    len(leak)
)

EMB: /kaggle/working/emb
PREP: /kaggle/input/datasets/sanawerb/preparation
3891 clips loaded from 31 chunks (full set = 3891)
Feature dimensions:
{'video (ViT mean)': 768, 'video (ViT mean+std)': 1536, 'audio (wav2vec2)': 768, 'audio (AST)': 768, 'audio (AST probs)': 527, 'video + wav2vec2': 1536, 'video + AST': 1536, 'video + AST + wav2vec2': 2304}

Split / matched counts:
split           test     train       val    
in_matched         0   1     0    1    0   1
cls                                         
Adult Content    119  99   411  251   42  26
Harmful Content   83  99   418  251  102  26
Safe             134  99   433  251   44  26
Suicide          139  99   418  251   44  26
test clips excluded for near-duplicate leakage: 1


In [16]:

# Cell 2: helpers (training, metrics, bootstrap)
def run_lr(Xs, ys, spl, shuffle=False):
    tr, va, te = [np.where(spl == s)[0] for s in ("train", "val", "test")]
    ytr = np.random.default_rng(SEED).permutation(ys[tr]) if shuffle else ys[tr]
    sc = StandardScaler().fit(Xs[tr])
    A, B, T = sc.transform(Xs[tr]), sc.transform(Xs[va]), sc.transform(Xs[te])
    best = None
    for C in (0.0005, 0.002, 0.01, 0.05, 0.3, 2):
        clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced").fit(A, ytr)
        f = f1_score(ys[va], clf.predict(B), average="macro")
        if best is None or f > best[0]: best = (f, C, clf)
    _, C, clf = best
    return dict(P=clf.predict_proba(T), y=ys[te], te=te, C=C)

def metrics(yt, P):
    pred = P.argmax(1)
    m = dict(macro_F1=f1_score(yt, pred, average="macro"),
             macro_AUC=roc_auc_score(yt, P, multi_class="ovr", average="macro", labels=[0, 1, 2, 3]),
             top2_acc=np.mean([yt[i] in np.argsort(-P[i])[:2] for i in range(len(yt))]))
    rec = recall_score(yt, pred, average=None, labels=[0, 1, 2, 3], zero_division=0)
    m.update({f"recall_{c}": r for c, r in zip(CLASS_NAMES, rec)})
    return m

def with_ci(yt, P, nboot=NBOOT):
    pt = metrics(yt, P); rng = np.random.default_rng(SEED); boots = []
    for _ in range(nboot):
        ix = rng.integers(0, len(yt), len(yt))
        if len(np.unique(yt[ix])) < 4: continue
        boots.append(metrics(yt[ix], P[ix]))
    b = pd.DataFrame(boots)
    return pt, {k: (np.percentile(b[k], 2.5), np.percentile(b[k], 97.5)) for k in pt}

def fmt(pt, ci, k): return f"{pt[k]:.3f} [{ci[k][0]:.3f}, {ci[k][1]:.3f}]"

In [17]:
# Cell 3: run all baselines on one set
def run_set(mask, name):
    Xm = {k: v[mask] for k, v in X.items()}; ym, sm = y[mask], split[mask]
    rows, preds = [], {}
    for fname, Xs in Xm.items():
        r = run_lr(Xs, ym, sm); pt, ci = with_ci(r["y"], r["P"]); preds[fname] = r
        rows.append(dict(features=fname, C=r["C"], macro_F1=fmt(pt, ci, "macro_F1"), macro_AUC=fmt(pt, ci, "macro_AUC"),
                         top2_acc=f"{pt['top2_acc']:.3f}", **{f"rec_{c}": round(pt[f'recall_{c}'], 2) for c in CLASS_NAMES}))
    # reference 1: metadata-only (the shortcut floor, same split, same metrics)
    mcols = [c for c in ["dur", "fps", "size_kb", "w", "h", "has_audio", "arate", "kbps", "motion", "contrast"] if c in man]
    M = man[mcols].fillna(0).values[mask]
    tr, te = np.where(sm == "train")[0], np.where(sm == "test")[0]
    g = GradientBoostingClassifier(random_state=SEED).fit(M[tr], ym[tr]); Pm = g.predict_proba(M[te])
    pt, ci = with_ci(ym[te], Pm)
    rows.append(dict(features="REFERENCE: metadata only (shortcut floor)", C="-", macro_F1=fmt(pt, ci, "macro_F1"),
                     macro_AUC=fmt(pt, ci, "macro_AUC"), top2_acc=f"{pt['top2_acc']:.3f}",
                     **{f"rec_{c}": round(pt[f'recall_{c}'], 2) for c in CLASS_NAMES}))
    # reference 2: shuffled training labels on the best-looking feature set (should sit near chance)
    r = run_lr(Xm["video + AST"], ym, sm, shuffle=True); pt, ci = with_ci(r["y"], r["P"], 200)
    rows.append(dict(features="SANITY: video + AST, shuffled labels", C="-", macro_F1=fmt(pt, ci, "macro_F1"),
                     macro_AUC=fmt(pt, ci, "macro_AUC"), top2_acc=f"{pt['top2_acc']:.3f}",
                     **{f"rec_{c}": round(pt[f'recall_{c}'], 2) for c in CLASS_NAMES}))
    res = pd.DataFrame(rows); pd.set_option("display.width", 250, "display.max_colwidth", 60)
    print(f"\n===== {name}: train={int((sm=='train').sum())} val={int((sm=='val').sum())} test={int((sm=='test').sum())} =====")
    print(res.to_string(index=False))
    res.to_csv(f"{RES}/baseline_{name}.csv", index=False)
    return res, preds, mask

res_m, preds_m, mask_m = run_set(in_matched, "matched")


===== matched: train=1004 val=104 test=395 =====
                                 features       C             macro_F1            macro_AUC top2_acc  rec_Safe  rec_Harmful  rec_Adult  rec_Suicide
                         video (ViT mean)    0.01 0.833 [0.793, 0.871] 0.951 [0.933, 0.968]    0.924      0.94         0.71       0.80         0.90
                     video (ViT mean+std)  0.0005 0.818 [0.779, 0.855] 0.945 [0.926, 0.962]    0.899      0.90         0.70       0.77         0.92
                         audio (wav2vec2)  0.0005 0.407 [0.358, 0.456] 0.681 [0.648, 0.715]    0.686      0.38         0.31       0.41         0.54
                              audio (AST)  0.0005 0.530 [0.482, 0.578] 0.795 [0.764, 0.826]    0.797      0.45         0.44       0.59         0.64
                        audio (AST probs)    0.01 0.499 [0.451, 0.549] 0.742 [0.707, 0.779]    0.770      0.45         0.41       0.58         0.56
                         video + wav2vec2  0.0005 0.816 [0.776

In [18]:
# Cell 4: confusion matrices (counts) for the key models on the matched test set
def show_cm(fname, preds):
    r = preds[fname]; cm = confusion_matrix(r["y"], r["P"].argmax(1), labels=[0, 1, 2, 3])
    print(f"\n{fname}  (rows = true, columns = predicted)")
    print(pd.DataFrame(cm, index=[f"true {c}" for c in CLASS_NAMES], columns=[f"pred {c}" for c in CLASS_NAMES]))
for f in ("video (ViT mean)", "audio (AST)", "video + AST"):
    show_cm(f, preds_m)


video (ViT mean)  (rows = true, columns = predicted)
              pred Safe  pred Harmful  pred Adult  pred Suicide
true Safe            93             1           3             2
true Harmful          9            70          11             9
true Adult           13             2          78             5
true Suicide          2             5           3            89

audio (AST)  (rows = true, columns = predicted)
              pred Safe  pred Harmful  pred Adult  pred Suicide
true Safe            45            19          20            15
true Harmful         23            44          20            12
true Adult           14            11          58            15
true Suicide         23             6           7            63

video + AST  (rows = true, columns = predicted)
              pred Safe  pred Harmful  pred Adult  pred Suicide
true Safe            92             4           1             2
true Harmful         13            68           8            10
true Adult      

In [19]:
# Cell 5: when does audio help? Split the matched test clips by what the audio contains
mm = man[mask_m].reset_index(drop=True)
te_ix = preds_m["video (ViT mean)"]["te"]                 # positions inside the matched subset
sub = mm.iloc[te_ix].reset_index(drop=True)
groups = {}
if "music_dominant" in sub and "speech_p" in sub:
    groups = {"music-dominant audio": sub.music_dominant.values == 1,
              "speech present (speech_p > 0.5)": sub.speech_p.values > 0.5,
              "other audio": (sub.music_dominant.values == 0) & (sub.speech_p.values <= 0.5)}
rows = []
for gname, g in groups.items():
    for fname in ("video (ViT mean)", "audio (AST)", "video + AST"):
        r = preds_m[fname]
        if g.sum() < 20: continue
        rows.append(dict(audio_type=gname, n=int(g.sum()), features=fname,
                         macro_F1=round(f1_score(r["y"][g], r["P"][g].argmax(1), average="macro"), 3)))
if rows:
    t = pd.DataFrame(rows).pivot(index=["audio_type", "n"], columns="features", values="macro_F1")
    print("macro-F1 by audio type (small groups: differences of a few points are noise)"); print(t)
    t.to_csv(f"{RES}/baseline_audio_type.csv")

macro-F1 by audio type (small groups: differences of a few points are noise)
features                             audio (AST)  video (ViT mean)  video + AST
audio_type                      n                                              
music-dominant audio            206        0.474             0.820        0.842
other audio                     61         0.567             0.878        0.822
speech present (speech_p > 0.5) 128        0.533             0.776        0.734


In [20]:
# Cell 6: secondary set (full clean set, same models) and summary
res_f, preds_f, _ = run_set(np.ones(n, bool), "full")
json.dump({"n_clips": int(n), "matched": res_m.to_dict("records"), "full": res_f.to_dict("records")},
          open(f"{RES}/baseline_summary.json", "w"), indent=2, default=str)
print(f"\nSaved to {RES}")


===== full: train=2684 val=336 test=870 =====
                                 features       C             macro_F1            macro_AUC top2_acc  rec_Safe  rec_Harmful  rec_Adult  rec_Suicide
                         video (ViT mean)   0.002 0.865 [0.840, 0.888] 0.968 [0.959, 0.977]    0.952      0.94         0.76       0.81         0.94
                     video (ViT mean+std)   0.002 0.860 [0.834, 0.882] 0.969 [0.960, 0.977]    0.957      0.93         0.77       0.82         0.92
                         audio (wav2vec2)   0.002 0.518 [0.485, 0.551] 0.779 [0.757, 0.801]    0.774      0.59         0.39       0.52         0.58
                              audio (AST)  0.0005 0.651 [0.619, 0.682] 0.850 [0.831, 0.870]    0.831      0.70         0.54       0.63         0.72
                        audio (AST probs)  0.0005 0.590 [0.556, 0.623] 0.801 [0.779, 0.824]    0.801      0.62         0.46       0.62         0.66
                         video + wav2vec2   0.002 0.857 [0.832, 0

In [23]:
!zip -r resultss.zip /kaggle/working/results

  adding: kaggle/working/results/ (stored 0%)
  adding: kaggle/working/results/baseline_full.csv (deflated 55%)
  adding: kaggle/working/results/baseline_summary.json (deflated 83%)
  adding: kaggle/working/results/baseline_audio_type.csv (deflated 23%)
  adding: kaggle/working/results/baseline_matched.csv (deflated 53%)


In [21]:
def f1_of(y_, P_, ix): return f1_score(y_[ix], P_[ix].argmax(1), average="macro")
def paired_diff(preds, base, other, nboot=2000):
    y_ = preds[base]["y"]; Pb, Po = preds[base]["P"], preds[other]["P"]
    rng = np.random.default_rng(0); d = []
    for _ in range(nboot):
        ix = rng.integers(0, len(y_), len(y_)); d.append(f1_of(y_, Po, ix) - f1_of(y_, Pb, ix))
    d = np.array(d); print(f"  {other:26s} minus {base}: {d.mean():+.3f}  95% CI [{np.percentile(d,2.5):+.3f}, {np.percentile(d,97.5):+.3f}]")

for name, pr in (("matched", preds_m), ("full", preds_f)):
    print(name)
    for o in ("video + AST", "video + wav2vec2", "video + AST + wav2vec2"):
        paired_diff(pr, "video (ViT mean)", o)

matched
  video + AST                minus video (ViT mean): -0.010  95% CI [-0.040, +0.018]
  video + wav2vec2           minus video (ViT mean): -0.017  95% CI [-0.047, +0.013]
  video + AST + wav2vec2     minus video (ViT mean): -0.008  95% CI [-0.037, +0.023]
full
  video + AST                minus video (ViT mean): +0.002  95% CI [-0.014, +0.019]
  video + wav2vec2           minus video (ViT mean): -0.008  95% CI [-0.024, +0.008]
  video + AST + wav2vec2     minus video (ViT mean): -0.009  95% CI [-0.028, +0.010]


In [22]:
def creator(path):
    b = os.path.splitext(os.path.basename(path))[0]
    m = re.match(r"^(.*)_(\d{12,20})$", b)
    return (m.group(1) if m else b).lower()
man["creator"] = man.path.map(creator)
print(man.creator.sample(8, random_state=0).tolist())
print("distinct creators:", man.creator.nunique(), "of", len(man), "clips")
cc = man.groupby("creator").cls.nunique()
print("creators with clips in >1 class:", int((cc > 1).sum()), "| creators with >1 clip:", int((man.creator.value_counts() > 1).sum()))

def creator_report(preds, mask, name):
    mm_ = man[mask].reset_index(drop=True); sp = split[mask]
    pool = set(mm_.creator[np.isin(sp, ["train", "val"])])
    sub_ = mm_.iloc[preds["video (ViT mean)"]["te"]].reset_index(drop=True)
    seen = sub_.creator.isin(pool).values
    print(f"\n{name}: test clips whose creator also appears in train/val: {seen.sum()} of {len(seen)}")
    rows = []
    for f in ("video (ViT mean)", "audio (AST)", "video + AST"):
        r = preds[f]
        for lab, g in (("creator seen", seen), ("creator NOT seen", ~seen)):
            if g.sum() >= 20:
                rows.append(dict(features=f, group=lab, n=int(g.sum()),
                                 macro_F1=round(f1_score(r["y"][g], r["P"][g].argmax(1), average="macro"), 3)))
    print(pd.DataFrame(rows).pivot(index="features", columns="group", values="macro_F1"))

creator_report(preds_m, mask_m, "matched"); creator_report(preds_f, np.ones(n, bool), "full")

['opsoc', 'zhitnikk', 'tmusicplus', 'renji_anime', 'dylanjackson230', 'magicmikelivefanclub', 'ngocxinkk47', 'huyenda2003']
distinct creators: 2529 of 3891 clips
creators with clips in >1 class: 15 | creators with >1 clip: 358

matched: test clips whose creator also appears in train/val: 0 of 395
group             creator NOT seen
features                          
audio (AST)                  0.530
video (ViT mean)             0.833
video + AST                  0.823

full: test clips whose creator also appears in train/val: 0 of 870
group             creator NOT seen
features                          
audio (AST)                  0.651
video (ViT mean)             0.865
video + AST                  0.867
